In [8]:
import pandas as frame
from patsy import origin

from src.pipelines.featurePipeline import FeaturePipeline


In [60]:
orig = frame.read_csv('/Users/macbookpro/platform/Backend/data/working/origination_v02.csv')

/var/folders/gy/2mbpv0kx5jz22fry28cqlnkc0000gn/T/ipykernel_57306/3542022219.py:1: DtypeWarning: Columns (25,26,30) have mixed types. Specify dtype option on import or set low_memory=False.
  orig = frame.read_csv('/Users/macbookpro/platform/Backend/data/working/origination_v02.csv')


In [9]:
hist  = frame.read_csv('/Users/macbookpro/platform/Backend/data/working/hist_12m.csv')

/var/folders/gy/2mbpv0kx5jz22fry28cqlnkc0000gn/T/ipykernel_57306/2861163649.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  hist  = frame.read_csv('/Users/macbookpro/platform/Backend/data/working/hist_12m.csv')


In [71]:
import importlib
import src.pipelines.window_builder as window_builder
importlib.reload(window_builder)

from src.pipelines.window_builder import WindowBuilder
from src.pipelines.delinquency_features import DelinquencyFeatures
from src.pipelines.capital_features import CapitalFeatures
from src.pipelines.origination_features import OriginationFeatures

In [ ]:
import time

t0 = time.time()
hist_12m = WindowBuilder(hist, window_months=12).build()
print(f"WindowBuilder      : {time.time()-t0:.1f}s")

t0 = time.time()
delinquency_agg = DelinquencyFeatures(hist_12m).build()
print(f"DelinquencyFeatures: {time.time()-t0:.1f}s")

t0 = time.time()
capital_agg = CapitalFeatures(hist_12m, orig_df=orig).build()
print(f"CapitalFeatures    : {time.time()-t0:.1f}s")

t0 = time.time()
orig_agg = OriginationFeatures(orig).build()
print(f"OriginationFeatures: {time.time()-t0:.1f}s")

In [11]:
import dask.dataframe as dd
import time

In [12]:
t0 = time.time()
data = dd.from_pandas(hist, npartitions=8)
print(f"Dask DataFrame     : {time.time()-t0:.1f}s")


Dask DataFrame     : 42.0s


In [13]:
t0 = time.time()
essai = hist.copy()
print(f"Pandas Copy        : {time.time()-t0:.1f}s")


Pandas Copy        : 4.8s


In [14]:
hist.head()

,LOAN_SEQUENCE_NUMBER,MONTHLY_REPORTING_PERIOD,CURRENT_ACTUAL_UPB,CURRENT_LOAN_DELINQUENCY_STATUS,LOAN_AGE,REMAINING_MONTHS_TO_LEGAL_MATURITY,MODIFICATION_FLAG,ZERO_BALANCE_CODE,ZERO_BALANCE_EFFECTIVE_DATE,CURRENT_INTEREST_RATE,CURRENT_NON_INTEREST_BEARING_UPB,DUE_DATE_OF_LAST_PAID_INSTALLMENT,INTEREST_RATE_STEP_INDICATOR,ESTIMATED_LTV,DELINQUENCY_DUE_TO_DISASTER,BORROWER_ASSISTANCE_STATUS_CODE,INTEREST_BEARING_UPB,VINTAGE,DPD_DAYS
0,F07Q10000001,2007-05-01,168000.0,0,1,239.0,N,0.0,NaN,7.375,0.0,NaN,N,74.556213,N,N,168000.0,2007Q1,0.0
1,F07Q10000001,2007-06-01,168000.0,0,2,238.0,N,0.0,NaN,7.375,0.0,NaN,N,74.556213,N,N,168000.0,2007Q1,0.0
2,F07Q10000001,2007-07-01,168000.0,0,3,237.0,N,0.0,NaN,7.375,0.0,NaN,N,74.556213,N,N,168000.0,2007Q1,0.0
3,F07Q10000001,2007-08-01,167000.0,0,4,236.0,N,0.0,NaN,7.375,0.0,NaN,N,74.112426,N,N,167000.0,2007Q1,0.0
4,F07Q10000001,2007-09-01,167000.0,0,5,235.0,N,0.0,NaN,7.375,0.0,NaN,N,74.112426,N,N,167000.0,2007Q1,0.0


In [41]:
t0 = time.time()
delinquency_agg = DelinquencyFeatures(hist).build()
print(f"DelinquencyFeatures: {time.time()-t0:.1f}s")

Copy DataFrame     : 6.8s
Cast DPD           : 6.5s
Groupby            : 0.0s
Colonnes de travail: 3.8s
DelinquencyFeatures: 30.9s


In [45]:
delinquency_agg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1390970 entries, 0 to 1390969
Data columns (total 11 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   LOAN_SEQUENCE_NUMBER   1390970 non-null  object 
 1   freq                   1390970 non-null  float64
 2   severite               1390970 non-null  float64
 3   profondeur_max         1390970 non-null  float64
 4   n_profondeur_max       1390970 non-null  int64  
 5   tendance               1390113 non-null  float64
 6   recuperation           89052 non-null    float64
 7   freq_x_profondeur_max  1390970 non-null  float64
 8   freq_x_tendance        1390113 non-null  float64
 9   freq_x_recuperation    89052 non-null    float64
 10  recidivisme_extreme    1390970 non-null  float64
dtypes: float64(9), int64(1), object(1)
memory usage: 116.7+ MB


In [63]:
t0 = time.time()
capital_agg = CapitalFeatures(hist, orig_df=orig).build()
print(f"CapitalFeatures    : {time.time()-t0:.1f}s")

CapitalFeatures    : 35.5s


In [64]:
capital_agg.head()

,LOAN_SEQUENCE_NUMBER,niveau,progression,ecart_au_plan,anticipation
0,F07Q10000001,0.000000,49621.425749,165152.653246,0.083333
1,F07Q10000002,0.000000,3.771474,0.000000,0.000000
2,F07Q10000003,0.861902,5.698937,-16660.654917,0.000000
3,F07Q10000004,0.000000,175.203789,8413.803387,0.083333
4,F07Q10000005,0.000000,33592.307319,113074.960468,0.083333


In [65]:
t0 = time.time()
orig_agg = OriginationFeatures(orig).build()
print(f"OriginationFeatures: {time.time()-t0:.1f}s")

OriginationFeatures: 2.9s


In [69]:
hist.shape

(15041114, 19)

In [68]:
window_builder = window_builder.WindowBuilder(hist).build()

In [70]:
window_builder.shape

(15041114, 19)

In [72]:
pipeline   = FeaturePipeline(window_months=12)
X, y       = pipeline.fit_transform(hist, orig)

Copy DataFrame     : 7.3s
Cast DPD           : 6.9s
Groupby            : 0.0s
Colonnes de travail: 9.8s


TypeError: check_array() got an unexpected keyword argument 'force_all_finite'. Did you mean 'ensure_all_finite'?